# Exploratory Data Analysis: Student Performance

This notebook explores the original student performance dataset using descriptive statistics and visualizations. It does not modify the raw data, encode or scale features, remove outliers, split the data, or train a model.

## 1. Import Libraries

These libraries support tabular inspection and basic statistical visualizations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Load Dataset

The raw CSV is loaded directly. All analysis below uses this dataframe without modifying it.

In [ ]:
data_path = "../data/raw/StudentPerformanceFactors.csv"
df = pd.read_csv(data_path)
target_column = "Exam_Score"

print(f"Loaded dataset with {df.shape[0]:,} rows and {df.shape[1]} columns.")

## 3. Dataset Overview

This section summarizes the dataset structure and separates numerical columns from categorical columns for descriptive analysis.

In [ ]:
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()
numerical_predictors = [column for column in numerical_columns if column != target_column]

print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame(name="dtype"))
print("Numerical columns:", numerical_columns)
print("Categorical columns:", categorical_columns)
print("\nDescriptive statistics:")
display(df.describe(include="all").transpose())

## 4. Target Variable Analysis

`Exam_Score` is treated as a continuous numerical target for descriptive analysis. Potential outliers are identified with the IQR rule only; they are not removed or changed.

In [ ]:
target = df[target_column]
target_summary = pd.Series({
    "minimum": target.min(),
    "maximum": target.max(),
    "mean": target.mean(),
    "median": target.median(),
    "standard_deviation": target.std(),
    "skewness": target.skew(),
})

print("Exam_Score statistics:")
display(target_summary.to_frame(name=target_column))

q1 = target.quantile(0.25)
q3 = target.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
potential_outliers = target[(target < lower_bound) | (target > upper_bound)]

print(f"IQR lower bound: {lower_bound:.2f}")
print(f"IQR upper bound: {upper_bound:.2f}")
print(f"Potential outlier count: {len(potential_outliers)}")
print("Potential outliers are reported only; none are removed.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(target, bins=20, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of Exam_Score")
axes[0].set_xlabel("Exam_Score")

sns.boxplot(x=target, ax=axes[1], color="lightcoral")
axes[1].set_title("Boxplot of Exam_Score")
axes[1].set_xlabel("Exam_Score")

plt.tight_layout()
plt.show()

## 5. Numerical Feature Analysis

Each numerical predictor is compared with `Exam_Score` using scatter plots and regression trend lines. The relationships are descriptive associations, not causal conclusions.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for axis, feature in zip(axes, numerical_predictors):
    sns.regplot(
        data=df,
        x=feature,
        y=target_column,
        ax=axis,
        scatter_kws={"alpha": 0.25, "s": 18},
        line_kws={"color": "darkred"},
    )
    axis.set_title(f"{feature} vs {target_column}")

plt.tight_layout()
plt.show()

feature_target_correlations = (
    df[numerical_predictors + [target_column]]
    .corr()[target_column]
    .drop(target_column)
    .sort_values(key=abs, ascending=False)
)

print("Numerical feature correlations with Exam_Score:")
display(feature_target_correlations.to_frame(name="correlation"))

## 6. Correlation Analysis

This heatmap shows pairwise Pearson correlations among numerical variables. Correlation describes linear association and does not establish causation.

In [ ]:
correlation_matrix = df[numerical_columns].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
)
plt.title("Correlation Matrix for Numerical Variables")
plt.tight_layout()
plt.show()

print("Selected correlations with Exam_Score:")
selected_features = [
    "Hours_Studied",
    "Attendance",
    "Previous_Scores",
    "Tutoring_Sessions",
    "Sleep_Hours",
    "Physical_Activity",
]
display(correlation_matrix.loc[selected_features, [target_column]])

## 7. Categorical Feature Analysis

The following compact grid compares `Exam_Score` distributions across all categorical features. These plots show group differences in the observed data; they do not imply causation.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes = axes.flatten()

for axis, feature in zip(axes, categorical_columns):
    sns.boxplot(
        data=df,
        x=feature,
        y=target_column,
        ax=axis,
        color="skyblue",
    )
    axis.set_title(feature)
    axis.set_xlabel("")
    axis.set_ylabel(target_column)
    axis.tick_params(axis="x", rotation=35)

for axis in axes[len(categorical_columns):]:
    axis.axis("off")

plt.tight_layout()
plt.show()

categorical_group_means = {
    feature: df.groupby(feature, dropna=False)[target_column]
    .agg(mean="mean", count="count")
    .sort_values("mean", ascending=False)
    for feature in categorical_columns
}

for feature, summary in categorical_group_means.items():
    print(f"{feature}:")
    display(summary)


## 8. Missing Value Visualization

Missing values are counted and visualized from the original dataframe. No missing values are filled or otherwise changed in this notebook.

In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]

print("Missing-value counts in the original dataset:")
display(missing_counts.to_frame(name="missing_count"))

plt.figure(figsize=(8, 4))
sns.barplot(x=missing_counts.values, y=missing_counts.index, color="darkorange")
plt.title("Missing Values by Column")
plt.xlabel("Missing count")
plt.ylabel("Column")
plt.tight_layout()
plt.show()

## 9. Key EDA Findings

- `Exam_Score` ranges from 55 to 101, with a mean of approximately 67.24, a median of 67.00, and a standard deviation of approximately 3.89.
- The target distribution is right-skewed in this dataset. Its skewness is approximately 1.65, so the mean is influenced by higher scores.
- The IQR rule flags 104 possible `Exam_Score` outliers using bounds of 59 and 75. These observations are retained for later investigation and are not removed here.
- Among the numerical predictors, `Attendance` has the strongest positive linear association with `Exam_Score` (correlation approximately 0.58), followed by `Hours_Studied` (approximately 0.45).
- `Previous_Scores` (approximately 0.18) and `Tutoring_Sessions` (approximately 0.16) show weaker positive linear associations with `Exam_Score`.
- `Physical_Activity` is nearly uncorrelated with `Exam_Score` (approximately 0.03), while `Sleep_Hours` shows a very weak negative correlation (approximately -0.02) in this dataset.
- Noticeable observed categorical group differences include `Access_to_Resources` (mean scores from approximately 66.20 to 68.09), `Parental_Involvement` (approximately 66.36 to 68.09), and `Learning_Disabilities` (approximately 66.27 for Yes versus 67.35 for No).
- `Peer_Influence`, `Motivation_Level`, `Family_Income`, and `Distance_from_Home` also show differences between some observed group means, while `Gender` and `School_Type` show very similar average scores.
- These are associations and group summaries in the observed data. They do not establish that any feature causes changes in `Exam_Score`.

## 10. Modeling Considerations

- `Exam_Score` is currently a continuous numerical target, so the dataset can support a regression formulation.
- The same target could potentially support a High/Average/Low classification formulation, but the project brief does not define the score thresholds for those categories.
- No High/Average/Low labels are created here, and no thresholds are invented.
- Before choosing between regression and classification, we need the agreed score thresholds, the intended prediction task, the evaluation metric, and confirmation that the resulting classes have useful and sufficiently balanced sample sizes.
- Model selection will also require a later train-test split and leakage-safe fitting of the preprocessing pipeline. No model is trained in this notebook.